<a href="https://colab.research.google.com/github/KejHo/Google-Colab/blob/main/gemma_4_12b_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma-4 12B – API server s pamětí a Skills

**Architektura:**
- Vlastní FastAPI server (OpenAI-kompatibilní endpoint)
- Paměť: JSON soubory na Google Drive
- Skills: SKILL.md soubory z Drive → automaticky do system promptu
- Cloudflare tunel → tvoje lokální AI se připojí jako k OpenAI API

**Struktura složek na Drive:**
```
MyDrive/
└── gemma-data/
    ├── memory.json       ← automaticky spravovaná paměť
    └── skills/
        ├── coding.md     ← libovolné SKILL.md soubory
        └── writing.md
```

**Jak model zapisuje do paměti:**
Kdykoli model chce uložit informaci, napíše do odpovědi tag:
`[REMEMBER: tato informace se uloží do paměti]`
Server tag automaticky extrahuje, uloží do JSON a odstraní z odpovědi.

In [ ]:
# ═══════════════════════════════════════════════════════════
# BUŇKA 1 – Instalace balíčků
# ═══════════════════════════════════════════════════════════
import psutil

def ram():
    m = psutil.virtual_memory()
    return f'{m.used/1e9:.1f}/{m.total/1e9:.1f} GB'

print(f'RAM před instalací: {ram()}')
print('Instaluji llama-cpp-python s CUDA podporou...')
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

print('Instaluji FastAPI + závislosti...')
!pip install -q huggingface_hub fastapi uvicorn sse-starlette psutil

print(f'RAM po instalaci: {ram()}')
print('✓ Hotovo – pokračuj buňkou 2.')

RAM před instalací: 1.0/13.6 GB
Instaluji llama-cpp-python s CUDA podporou...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 599.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
Instaluji FastAPI + závislosti...
RAM po instalaci: 1.0/13.6 GB
✓ Hotovo – pokračuj buňkou 2.


In [ ]:
# ═══════════════════════════════════════════════════════════
# BUŇKA 2 – Stažení modelu
# (SOBĚSTAČNÁ – funguje i po reconnectu)
# ═══════════════════════════════════════════════════════════
import os, psutil
from google.colab import userdata
from huggingface_hub import hf_hub_download

def ram():
    m = psutil.virtual_memory()
    return f'{m.used/1e9:.1f}/{m.total/1e9:.1f} GB'

HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise ValueError('HF_TOKEN není nastaven v Colab Secrets!')
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

print(f'RAM: {ram()}')
print('Stahuji model (~7.4 GB) – jen při prvním spuštění, pak z cache...')

model_path = hf_hub_download(
    repo_id='OBLITERATUS/Gemma-4-12B-OBLITERATED',
    filename='Gemma-4-12B-OBLITERATED-Q4_K_M.gguf',
    token=HF_TOKEN,
)

# Uložíme cestu – ostatní buňky ji načtou bez závislosti na proměnné
with open('/tmp/model_path.txt', 'w') as f:
    f.write(model_path)

print(f'✓ Model připraven: {model_path}')
print(f'RAM: {ram()}')
print('Pokračuj buňkou 3.')

RAM: 0.9/13.6 GB
Stahuji model (~7.4 GB) – jen při prvním spuštění, pak z cache...


Gemma-4-12B-OBLITERATED-Q4_K_M.gguf:   0%|          | 0.00/7.38G [00:00<?, ?B/s]

✓ Model připraven: /root/.cache/huggingface/hub/models--OBLITERATUS--Gemma-4-12B-OBLITERATED/snapshots/f81b0cbd28a3650138635823bc101adb56a0bc4a/Gemma-4-12B-OBLITERATED-Q4_K_M.gguf
RAM: 1.4/13.6 GB
Pokračuj buňkou 3.


In [ ]:
# ═══════════════════════════════════════════════════════════
# BUŇKA 3 – Příprava Drive, paměti a Skills
# (SOBĚSTAČNÁ – funguje i po reconnectu)
# ═══════════════════════════════════════════════════════════
import os, json
from google.colab import drive

print('Připojuji Google Drive...')
drive.mount('/content/drive')

DRIVE_BASE   = '/content/drive/MyDrive/gemma-data'
MEMORY_FILE  = f'{DRIVE_BASE}/memory.json'
SKILLS_DIR   = f'{DRIVE_BASE}/skills'

os.makedirs(SKILLS_DIR, exist_ok=True)

# Vytvoříme memory.json pokud neexistuje
if not os.path.exists(MEMORY_FILE):
    with open(MEMORY_FILE, 'w') as f:
        json.dump({'memories': []}, f, ensure_ascii=False, indent=2)
    print('✓ Vytvořen nový memory.json')
else:
    with open(MEMORY_FILE) as f:
        mem = json.load(f)
    print(f'✓ Načtena paměť: {len(mem["memories"])} vzpomínek')

# Ukážeme existující Skills
skills = [f for f in os.listdir(SKILLS_DIR) if f.endswith('.md')]
if skills:
    print(f'✓ Nalezeny Skills: {skills}')
else:
    # Vytvoříme ukázkový skill
    with open(f'{SKILLS_DIR}/example.md', 'w') as f:
        f.write('# Example Skill\n\nBuď stručný a přesný. Odpovídej v jazyce uživatele.')
    print('✓ Vytvořen ukázkový skill (example.md)')
    print(f'  Přidej vlastní .md soubory do: {SKILLS_DIR}')

# Uložíme cesty pro server
with open('/tmp/drive_paths.json', 'w') as f:
    json.dump({'memory': MEMORY_FILE, 'skills': SKILLS_DIR}, f)

print('\n✓ Drive připraven – pokračuj buňkou 4.')

Připojuji Google Drive...
Mounted at /content/drive
✓ Načtena paměť: 1 vzpomínek
✓ Nalezeny Skills: ['example.md']

✓ Drive připraven – pokračuj buňkou 4.


In [ ]:
import subprocess, time, psutil

def ram():
    m = psutil.virtual_memory()
    return f'{m.used/1e9:.1f}/{m.total/1e9:.1f} GB'

# Zapíšeme server skript
server_code = r'''
import os, json, re, time, threading
from datetime import datetime
from typing import List, Optional
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from llama_cpp import Llama

# ── Načtení konfigurace ──────────────────────────────────
with open("/tmp/model_path.txt") as f:
    MODEL_PATH = f.read().strip()

with open("/tmp/drive_paths.json") as f:
    paths = json.load(f)
MEMORY_FILE = paths["memory"]
SKILLS_DIR  = paths["skills"]

# ── Načtení modelu ───────────────────────────────────────
print("Načítám model...", flush=True)
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=8192,
    verbose=False,
)
print("✓ Model načten.", flush=True)

# ── Paměť ────────────────────────────────────────────────
mem_lock = threading.Lock()

def load_memory():
    try:
        with open(MEMORY_FILE) as f:
            return json.load(f).get("memories", [])
    except Exception:
        return []

def save_memory(memories):
    with mem_lock:
        with open(MEMORY_FILE, "w") as f:
            json.dump({"memories": memories}, f, ensure_ascii=False, indent=2)

def extract_and_save_memories(text: str) -> str:
    """Najde [REMEMBER: ...] tagy, uloží je do JSON, odstraní z textu."""
    tags = re.findall(r"\[REMEMBER:\s*(.+?)\]", text, re.IGNORECASE | re.DOTALL)
    if tags:
        memories = load_memory()
        for tag in tags:
            memories.append({
                "timestamp": datetime.now().isoformat(),
                "content": tag.strip()
            })
            # Zachováme max 200 vzpomínek
            if len(memories) > 200:
                memories = memories[-200:]
        save_memory(memories)
        # Odstraníme tagy z odpovědi
        text = re.sub(r"\[REMEMBER:\s*.+?\]", "", text, flags=re.IGNORECASE | re.DOTALL).strip()
    return text

# ── Skills ───────────────────────────────────────────────
def load_skills() -> str:
    """Načte všechny .md soubory ze Skills složky."""
    parts = []
    try:
        for fname in sorted(os.listdir(SKILLS_DIR)):
            if fname.endswith(".md"):
                with open(os.path.join(SKILLS_DIR, fname)) as f:
                    parts.append(f"### {fname}\n{f.read().strip()}")
    except Exception:
        pass
    return "\n\n".join(parts)

# ── System prompt builder ────────────────────────────────
def build_system_prompt(custom_system: str = "") -> str:
    sections = []

    # Skills
    skills_text = load_skills()
    if skills_text:
        sections.append(f"## SKILLS\n{skills_text}")

    # Paměť
    memories = load_memory()
    if memories:
        mem_lines = "\n".join(
            f"- [{m['timestamp'][:10]}] {m['content']}" for m in memories[-50:]
        )
        sections.append(f"## MEMORY (naposled 50 vzpomínek)\n{mem_lines}")

    # Instrukce pro paměť
    sections.append(
        "## INSTRUKCE PRO PAMĚŤ\n"
        "Pokud zjistíš důležitou informaci o uživateli nebo kontextu, "
        "zapiš ji do odpovědi jako: [REMEMBER: popis informace]\n"
        "Tagy budou automaticky uloženy a odstraněny z výstupu."
    )

    # Vlastní system prompt od klienta
    if custom_system:
        sections.append(f"## POKYNY OD KLIENTA\n{custom_system}")

    return "\n\n".join(sections)

# ── FastAPI ──────────────────────────────────────────────
app = FastAPI(title="Gemma-4 12B API")

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: Optional[str] = "gemma"
    messages: List[Message]
    max_tokens: Optional[int] = 2048
    temperature: Optional[float] = 0.7
    top_p: Optional[float] = 0.95
    stream: Optional[bool] = False

@app.get("/")
def root():
    return {"status": "online", "model": "Gemma-4-12B"}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": "gemma", "object": "model"}]}

@app.get("/memory")
def get_memory():
    """Zobrazí aktuální paměť."""
    return {"memories": load_memory()}

@app.delete("/memory")
def clear_memory():
    """Smaže celou paměť."""
    save_memory([])
    return {"status": "cleared"}

@app.get("/skills")
def get_skills():
    """Zobrazí načtené Skills."""
    return {"skills": load_skills()}

@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    # Separujeme system prompt od zbytku zpráv
    messages = [m.dict() for m in req.messages]
    custom_system = ""
    filtered = []
    for msg in messages:
        if msg["role"] == "system":
            custom_system += msg["content"] + "\n"
        else:
            filtered.append(msg)

    # Sestavíme systémový prompt (skills + paměť + instrukce)
    system_prompt = build_system_prompt(custom_system.strip())
    final_messages = [{"role": "system", "content": system_prompt}] + filtered

    try:
        result = llm.create_chat_completion(
            messages=final_messages,
            max_tokens=req.max_tokens,
            temperature=req.temperature,
            top_p=req.top_p,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

    raw_content = result["choices"][0]["message"]["content"]

    # Zpracování [REMEMBER:] tagů
    clean_content = extract_and_save_memories(raw_content)

    # OpenAI-kompatibilní odpověď
    return {
        "id": f"chatcmpl-{int(time.time())}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": clean_content},
            "finish_reason": result["choices"][0].get("finish_reason", "stop")
        }],
        "usage": result.get("usage", {})
    }

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open('/tmp/server.py', 'w') as f:
    f.write(server_code)
print('✓ Server skript zapsán.')

# Zastavíme starý server
subprocess.run('pkill -f server.py || true', shell=True)
subprocess.run('fuser -k 8000/tcp || true', shell=True)
time.sleep(2)

print(f'RAM před spuštěním: {ram()}')
print('Spouštím API server (načítání modelu zabere ~30–60 s)...')

with open('/tmp/server.log', 'w') as log_f:
    subprocess.Popen(
        ['python', '/tmp/server.py'],
        stdout=log_f, stderr=subprocess.STDOUT,
        start_new_session=True
    )

# Čekáme dokud server neodpoví
server_ok = False
for i in range(24):
    time.sleep(5)
    result = subprocess.run('curl -s http://localhost:8000/', shell=True, capture_output=True, text=True)
    if result.stdout:
        server_ok = True
        break
    print(f'  {(i+1)*5}s – RAM: {ram()}')

if server_ok:
    print(f'\n✓ API server běží! RAM: {ram()}')
    with open('/tmp/server_ok.txt', 'w') as f: f.write('ok')
    print('Pokračuj buňkou 5.')
else:
    print('\n✗ Server neodpovídá. Log:')
    with open('/tmp/server.log') as f: print(f.read()[-3000:])

✓ Server skript zapsán.
RAM před spuštěním: 1.4/13.6 GB
Spouštím API server (načítání modelu zabere ~30–60 s)...
  5s – RAM: 1.7/13.6 GB
  10s – RAM: 1.7/13.6 GB
  15s – RAM: 1.7/13.6 GB
  20s – RAM: 1.7/13.6 GB
  25s – RAM: 1.7/13.6 GB
  30s – RAM: 1.8/13.6 GB

✓ API server běží! RAM: 1.8/13.6 GB
Pokračuj buňkou 5.


In [ ]:
# ═══════════════════════════════════════════════════════════
# BUŇKA 5 – Cloudflare tunel → veřejná URL pro tvoji lokální AI
# (SOBĚSTAČNÁ – funguje i po reconnectu)
# ═══════════════════════════════════════════════════════════
import re, time, subprocess, os

cloudflared = '/tmp/cloudflared'
if not os.path.exists(cloudflared):
    print('Stahuji cloudflared...')
    subprocess.run(
        f'wget -q -O {cloudflared} '
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
        f'&& chmod +x {cloudflared}',
        shell=True
    )

subprocess.run("pkill -f 'cloudflared.*8000' || true", shell=True)
time.sleep(1)

print('Spouštím Cloudflare tunel...')
with open('/tmp/cloudflared.log', 'w') as log_f:
    subprocess.Popen(
        [cloudflared, 'tunnel', '--url', 'http://localhost:8000'],
        stdout=log_f, stderr=subprocess.STDOUT, start_new_session=True
    )

public_url = None
for i in range(60):
    time.sleep(3)
    try:
        with open('/tmp/cloudflared.log') as f:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', f.read())
        if match:
            public_url = match.group(0)
            break
    except Exception: pass
    if i % 5 == 4: print(f'  ...čekám ({(i+1)*3}s)')

print()
if public_url:
    print('=' * 65)
    print('✓ VŠE HOTOVO!')
    print()
    print(f'  API URL:       {public_url}')
    print(f'  Chat endpoint: {public_url}/v1/chat/completions')
    print(f'  Paměť:         {public_url}/memory')
    print(f'  Skills:        {public_url}/skills')
    print()
    print('  Nastavení v tvé lokální AI:')
    print(f'    Base URL:  {public_url}/v1')
    print('    API Key:   local  (nebo cokoli, klíč se neověřuje)')
    print('    Model:     gemma')
    print('=' * 65)
else:
    print('✗ URL se nepodařilo získat.')
    with open('/tmp/cloudflared.log') as f: print(f.read())

Stahuji cloudflared...
Spouštím Cloudflare tunel...

✓ VŠE HOTOVO!

  API URL:       https://data-push-gary-conclusion.trycloudflare.com
  Chat endpoint: https://data-push-gary-conclusion.trycloudflare.com/v1/chat/completions
  Paměť:         https://data-push-gary-conclusion.trycloudflare.com/memory
  Skills:        https://data-push-gary-conclusion.trycloudflare.com/skills

  Nastavení v tvé lokální AI:
    Base URL:  https://data-push-gary-conclusion.trycloudflare.com/v1
    API Key:   local  (nebo cokoli, klíč se neověřuje)
    Model:     gemma


In [ ]:
# ═══════════════════════════════════════════════════════════
# BUŇKA 6 – Test API + zobrazení paměti (volitelné)
# ═══════════════════════════════════════════════════════════
import subprocess, json

print('=== Test chat API ===')
result = subprocess.run(
    '''curl -s http://localhost:8000/v1/chat/completions \
    -H "Content-Type: application/json" \
    -d '{"messages":[{"role":"user","content":"Řekni ahoj a zapamatuj si, že mě zajímá Python."}]}'  ''',
    shell=True, capture_output=True, text=True
)
try:
    resp = json.loads(result.stdout)
    print('Odpověď:', resp['choices'][0]['message']['content'])
except Exception:
    print('Raw:', result.stdout[:500])

print('\n=== Aktuální paměť ===')
mem_result = subprocess.run('curl -s http://localhost:8000/memory', shell=True, capture_output=True, text=True)
try:
    mem = json.loads(mem_result.stdout)
    if mem['memories']:
        for m in mem['memories']:
            print(f"  [{m['timestamp'][:10]}] {m['content']}")
    else:
        print('  (prázdná paměť)')
except Exception:
    print(mem_result.stdout)

print('\n=== Načtené Skills ===')
skills_result = subprocess.run('curl -s http://localhost:8000/skills', shell=True, capture_output=True, text=True)
try:
    skills = json.loads(skills_result.stdout)
    print(skills['skills'][:500] + '...' if len(skills['skills']) > 500 else skills['skills'])
except Exception:
    print(skills_result.stdout)

=== Test chat API ===
Odpověď: Ahoj!

=== Aktuální paměť ===
  [2026-06-14] Uživatele zajímá Python
  [2026-06-16] Uživatele zajímá Python

=== Načtené Skills ===
### example.md
# Example Skill

Buď stručný a přesný. Odpovídej v jazyce uživatele.
